# 01 — Basic usage

OGC API - Environmental Data Retrieval (EDR) is a query-API for
environmental data (weather, ocean, climate, air quality). The 1.1 spec
defines several query types — `edr-xarray` implements the `/cubes`
query, which returns a regular grid as CoverageJSON.

This notebook walks through the smallest possible workflow:

1. Discover what collections the server exposes.
2. Open one collection as a lazy `xarray.Dataset`.
3. Inspect dimensions, variables, coordinates, attributes.
4. Trigger a single lazy fetch by reading `.values`.
5. Close the dataset when finished.

In [7]:
%pip install -q -e ..

Note: you may need to restart the kernel to use updated packages.


## 1. Discover collections

Every EDR server exposes `GET /collections` — a catalogue of everything
it serves. One HTTP call is all it takes to see what's available.

In [8]:
import httpx

server = "http://127.0.0.1:8000"  # replace with your EDR server root

resp = httpx.get(f"{server}/collections")
resp.raise_for_status()

collections = resp.json()["collections"]
for c in collections:
    print(c["id"], "-", c.get("title", ""))

EO:EUM:DAT:0398 - msg_frm


In [9]:
# Pick the first collection (or set collection_id to any id from the list above)
collection_id = collections[0]["id"]
collection_url = f"{server}/collections/{collection_id}"
print(collection_url)

http://127.0.0.1:8000/collections/EO:EUM:DAT:0398


## 2. Open the collection

`edr-xarray` registers itself as the `"edr"` xarray engine when imported.
Pointing `xr.open_dataset` at a `/collections/{id}` URL with
`engine="edr"` is all it takes.

In [10]:
import xarray as xr

import edr_xarray  # registers engine="edr"

ds = xr.open_dataset(
    collection_url,
    engine="edr",
)
ds

<xarray.Dataset> Size: 30kB
Dimensions:  (t: 746, y: 2, x: 2)
Coordinates:
  * t        (t) datetime64[ns] 6kB 2024-01-01T12:00:00 ... 2026-01-15T12:00:00
  * y        (y) float64 16B 34.0 34.1
  * x        (x) float64 16B -47.0 -46.9
Data variables:
    FWI      (t, y, x) float64 24kB ...
Attributes:
    Conventions:  CF-1.10
    title:        msg_frm
    summary:      FireCube product 'lsasaf-fire-risk-map.msg-seviri-0-degree....

## 3. Inspect structure (no fetch)

At this point the library has issued **at most two requests**: one for
the collection metadata, plus an optional probe of the cube endpoint to
discover grid axes. No actual data values have been transferred yet.

In [11]:
print("dims:      ", dict(ds.sizes))
print("data_vars: ", list(ds.data_vars))
print("coords:    ", list(ds.coords))
print("attrs:     ", dict(ds.attrs))

dims:       {'t': 746, 'y': 2, 'x': 2}
data_vars:  ['FWI']
coords:     ['t', 'y', 'x']
attrs:      {'Conventions': 'CF-1.10', 'title': 'msg_frm', 'summary': "FireCube product 'lsasaf-fire-risk-map.msg-seviri-0-degree.level-2.zarr' for plugin 'msg_frm'."}


## 4. Lazy load values

Calling `.values` (or `.load()`, `.compute()`) on a `DataArray` triggers
a single GET against the cube endpoint. With no slicing, the full grid
is requested.

In [12]:
var = next(iter(ds.data_vars))  # first variable
arr = ds[var].values
print("variable:", var)
print("shape:   ", arr.shape)
print("dtype:   ", arr.dtype)

EdrServerError: Cube selection has 451873045 cells, exceeding limit of 1000000 [status=400, url=http://127.0.0.1:8000/collections/EO:EUM:DAT:0398/cube?f=CoverageJSON&datetime=2024-01-01T12%3A00%3A00Z%2F2026-01-15T12%3A00%3A00Z&bbox=-47.0%2C34.0%2C79.0%2C82.0&parameter-name=FWI]

## 5. Cleanup

Always close the dataset when you are done.

In [ ]:
ds.close()